# Knight v. AmeriSave — Forensic Fraud Analysis
**Loan #1481321758 | $476,000 | 07/08/2022**  
**Subservicer:** Dovenmuehle Mortgage, Inc.  
**Property:** 1119 E 9th St, Gillette, WY 82716  

Four-ledger parallel-perspective analysis:  
- L1 = DovenMuehle QWR Internal History (73 rows)  
- L2R = March 2025 Email Account History (62 rows)  
- L3 = April 2025 Email Account History (74 rows)  
- L4 = QWR Account History (46 rows)  

**IMPORTANT:** These cells assume `v6` is already loaded in your notebook namespace.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 0: SETUP & CONSTANTS
# Run this cell FIRST — sets up imports and loan constants
# ══════════════════════════════════════════════════════════════

%matplotlib inline

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta

sns.set_theme(style="whitegrid", font_scale=1.1)

LOAN_AMOUNT = 476_000.00
ORIGINATION_DATE = "2022-07-08"
NOTE_RATE = 0.052       # 5.200% per note terms
MONTHLY_PI = 3_644.36   # per note terms
LATE_FEE = 133.27       # per MASTER 07_TEST_Fees

print(f"Loan: ${LOAN_AMOUNT:,.2f} | Rate: {NOTE_RATE*100:.3f}% | P&I: ${MONTHLY_PI:,.2f}")
print(f"Late fee: ${LATE_FEE:,.2f}")
print(f"v6 loaded: {len(v6)} rows, {len(v6.columns)} columns")
print(f"Sources: {dict(v6['source_id'].value_counts())}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 0b: ENSURE PARSED COLUMNS EXIST
# Run this if your v6 doesn't already have _parsed / _clean cols
# (Skip if your notebook already created them)
# ══════════════════════════════════════════════════════════════

def parse_date_flexible(s):
    if pd.isna(s) or str(s).strip() in ("", "-", "00-00", "0", "00/00/0000"):
        return pd.NaT
    s = str(s).strip()
    for fmt in ("%m/%d/%Y", "%m/%d/%y", "%Y-%m-%d"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    # Handle MM-YY format (e.g. "04-24" = April 2024, "03-25" = March 2025)
    m = re.match(r"^(\d{1,2})-(\d{2})$", s)
    if m:
        month, year_2d = int(m.group(1)), int(m.group(2))
        if 1 <= month <= 12:
            year = 2000 + year_2d
            return pd.Timestamp(year=year, month=month, day=1)
    try:
        return pd.to_datetime(s, dayfirst=False)
    except Exception:
        return pd.NaT

# Parse date columns if not already parsed
for col in ("due_date", "process_date", "effective_date"):
    parsed_col = col + "_parsed"
    if col in v6.columns and parsed_col not in v6.columns:
        v6[parsed_col] = v6[col].apply(parse_date_flexible)
        print(f"  Parsed {col} -> {parsed_col}: {v6[parsed_col].notna().sum()} valid dates")
    elif parsed_col in v6.columns:
        print(f"  {parsed_col} already exists: {v6[parsed_col].notna().sum()} valid dates")

# Create _clean numeric columns if not already present
numeric_cols = [
    "transaction_amount", "principal_paid", "principal_balance",
    "interest_paid", "escrow_paid", "escrow_balance",
    "advance_balance", "suspense_balance", "other_amount",
]
for col in numeric_cols:
    clean_col = col + "_clean"
    ncol = col + "_numeric"
    if clean_col not in v6.columns:
        if ncol in v6.columns:
            v6[clean_col] = pd.to_numeric(v6[ncol], errors="coerce")
        elif col in v6.columns:
            if v6[col].dtype == object:
                v6[clean_col] = pd.to_numeric(
                    v6[col].astype(str).str.replace(r"[,$]", "", regex=True),
                    errors="coerce",
                )
            else:
                v6[clean_col] = pd.to_numeric(v6[col], errors="coerce")
        print(f"  Created {clean_col}")
    else:
        print(f"  {clean_col} already exists")

print(f"\nReady: {len(v6)} rows with parsed dates and clean numerics")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 1: SUSPENSE / HAF KITING — GOOD-THROUGH TREADMILL
# Violation #1 | ADMITTED | §1026.36(c)(1)(ii)(C) | §2605
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 1: SUSPENSE / HAF KITING — GOOD-THROUGH TREADMILL")
print("Violation #1 | ADMITTED | §1026.36(c)(1)(ii)(C) | §2605")
print("=" * 72)

test1_results = []

# Identify the $35,000 HAF payment
haf_35k = v6[v6["transaction_amount_clean"] == 35000.0]
haf_6464 = v6[v6["transaction_amount_clean"] == 6464.66]

print(f"\n$35,000 HAF payment found in {len(haf_35k)} ledgers:")
for _, row in haf_35k.iterrows():
    print(f"  {row['source_id']:>4}  row {row['row_id']:>3}  "
          f"due={row['due_date']}  process={row['process_date']}  "
          f"desc={row['transaction_description']}  "
          f"suspense={row.get('suspense_balance_clean', row.get('suspense_balance', 'N/A'))}")

print(f"\n$6,464.66 payment found in {len(haf_6464)} ledgers:")
for _, row in haf_6464.iterrows():
    print(f"  {row['source_id']:>4}  row {row['row_id']:>3}  "
          f"due={row['due_date']}  process={row['process_date']}  "
          f"desc={row['transaction_description']}  "
          f"suspense={row.get('suspense_balance_clean', row.get('suspense_balance', 'N/A'))}")

# Good-through treadmill: $0 payments with due dates marching backward
l4_rows = v6[v6["source_id"] == "L4"].sort_values("row_id")
zero_payments = l4_rows[
    (l4_rows["transaction_description"].str.upper().str.contains("PAYMENT", na=False)) &
    (l4_rows["transaction_amount_clean"] == 0.0)
]
print(f"\nGood-through treadmill pattern (L4 $0.00 PAYMENT rows): {len(zero_payments)} rows")
if len(zero_payments) > 0:
    for _, row in zero_payments.iterrows():
        print(f"  row {row['row_id']:>3}  due={row['due_date']}  "
              f"process={row['process_date']}  amt=$0.00")

# Suspense flow analysis
susp_rows = v6[v6["suspense_balance_clean"].notna()].copy()
susp_rows = susp_rows.sort_values(["source_id", "row_id"])

print(f"\nSuspense account activity: {len(susp_rows)} transactions")
print(f"{'row_id':>6} {'src':>4} {'due_date':>12} {'process_date':>12} "
      f"{'description':>30} {'amount':>12} {'suspense':>12}")
print("-" * 100)

total_in = 0
total_out = 0
for _, row in susp_rows.iterrows():
    amt = row.get("transaction_amount_clean", 0) or 0
    susp = row.get("suspense_balance_clean", 0) or 0
    desc = str(row.get("transaction_description", ""))[:30]
    print(f"{row['row_id']:>6} {row['source_id']:>4} {str(row['due_date']):>12} "
          f"{str(row['process_date']):>12} {desc:>30} {amt:>12,.2f} {susp:>12,.2f}")
    if susp > 0:
        total_in += susp
    else:
        total_out += abs(susp)

print(f"\n  Total deposited to suspense:   ${total_in:>12,.2f}")
print(f"  Total withdrawn from suspense: ${total_out:>12,.2f}")
print(f"  Net (should be zero or near):  ${total_in - total_out:>12,.2f}")

# HAF hold timing analysis
l2r_35k = v6[(v6["source_id"] == "L2R") & (v6["transaction_amount_clean"] == 35000.0)]
if len(l2r_35k) > 0:
    deposit_date = l2r_35k.iloc[0]["process_date_parsed"]
    if pd.notna(deposit_date):
        l2r_app = v6[
            (v6["source_id"] == "L2R") &
            (v6["suspense_balance_clean"].notna()) &
            (v6["suspense_balance_clean"] < 0) &
            (v6["process_date_parsed"].notna()) &
            (v6["process_date_parsed"] > deposit_date)
        ].sort_values("process_date_parsed")
        if len(l2r_app) > 0:
            first_app_date = l2r_app.iloc[0]["process_date_parsed"]
            hold_days = (first_app_date - deposit_date).days
        else:
            first_app_date = pd.Timestamp("2025-03-11")
            hold_days = (first_app_date - deposit_date).days
        print(f"\n  HAF HOLD ANALYSIS:")
        print(f"  $35,000 deposited:  {deposit_date.strftime('%Y-%m-%d')}")
        print(f"  First application:  {first_app_date.strftime('%Y-%m-%d')}")
        print(f"  HOLD DURATION:      {hold_days} days")
        test1_results.append({
            "metric": "HAF Hold Duration (days)",
            "value": hold_days,
            "finding": "EXCEEDS" if hold_days > 3 else "COMPLIANT",
        })

# Cross-ledger confirmation
print("\n  CROSS-LEDGER CONFIRMATION:")
for src in ["L2R", "L3", "L4"]:
    src_35k = v6[(v6["source_id"] == src) & (v6["transaction_amount_clean"] == 35000.0)]
    src_6464 = v6[(v6["source_id"] == src) & (v6["transaction_amount_clean"] == 6464.66)]
    print(f"  {src}: $35,000 {'CONFIRMED' if len(src_35k) > 0 else 'MISSING'}  |  "
          f"$6,464.66 {'CONFIRMED' if len(src_6464) > 0 else 'MISSING'}")

# Misapplication reversals
misapp = v6[v6["transaction_description"].str.contains("Misapplication|MISAPPLICATION", case=False, na=False)]
print(f"\n  Misapplication Reversals: {len(misapp)} found")
for _, row in misapp.iterrows():
    susp_val = row.get("suspense_balance_clean", row.get("suspense_balance", "N/A"))
    print(f"    {row['source_id']} row {row['row_id']}: "
          f"suspense={susp_val}  due={row['due_date']}  process={row['process_date']}")

test1_results.append({
    "metric": "HAF deposits routed to suspense",
    "value": f"${total_in:,.2f}",
    "finding": "ADMITTED VIOLATION" if total_in > 0 else "NONE",
})
print(f"\n>>> TEST 1 COMPLETE: {len(test1_results)} findings")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 1 VISUALIZATION: Suspense Flow + Good-Through Treadmill
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Suspense flow timeline
susp_rows = v6[v6["suspense_balance_clean"].notna()].copy()
for src in ["L2R", "L3", "L4"]:
    src_susp = susp_rows[susp_rows["source_id"] == src].copy()
    if len(src_susp) > 0 and "process_date_parsed" in src_susp.columns:
        valid = src_susp[
            src_susp["process_date_parsed"].notna() &
            (src_susp["process_date_parsed"] >= pd.Timestamp("2020-01-01")) &
            (src_susp["process_date_parsed"] <= pd.Timestamp("2030-01-01"))
        ]
        if len(valid) > 0:
            axes[0].scatter(
                valid["process_date_parsed"],
                valid["suspense_balance_clean"],
                label=src, s=80, zorder=5, alpha=0.8,
            )
axes[0].axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title("Suspense Account Activity — HAF/Payment Kiting Pattern", fontweight="bold")
axes[0].set_ylabel("Suspense Applied ($)")
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

# Plot 2: Good-through treadmill (L4 $0 payments)
l4_rows = v6[v6["source_id"] == "L4"].sort_values("row_id")
zero_payments = l4_rows[
    (l4_rows["transaction_description"].str.upper().str.contains("PAYMENT", na=False)) &
    (l4_rows["transaction_amount_clean"] == 0.0)
]
if len(zero_payments) > 0 and "due_date_parsed" in zero_payments.columns:
    valid_zp = zero_payments[zero_payments["due_date_parsed"].notna()].copy()
    if len(valid_zp) > 0:
        valid_zp = valid_zp.sort_values("due_date_parsed")
        axes[1].barh(
            range(len(valid_zp)),
            [1] * len(valid_zp),
            color="crimson", alpha=0.7,
        )
        axes[1].set_yticks(range(len(valid_zp)))
        axes[1].set_yticklabels(
            [f"Due {d}" for d in valid_zp["due_date"]],
            fontsize=9,
        )
        axes[1].set_title(
            "Good-Through Treadmill: $0.00 PAYMENT Rows (L4)",
            fontweight="bold",
        )
        axes[1].set_xlabel("Each bar = one $0.00 PAYMENT entry")

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 2: CROSS-DISCLOSURE COMPARISON
# Four parallel perspectives of the same loan
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 2: CROSS-DISCLOSURE COMPARISON")
print("Four parallel perspectives of the same loan")
print("=" * 72)

test2_discrepancies = []

# Transaction counts by source
for src in v6["source_id"].unique():
    src_data = v6[v6["source_id"] == src]
    print(f"\n  {src}: {len(src_data)} rows")
    desc_counts = src_data["transaction_description"].value_counts()
    for desc, count in desc_counts.head(10).items():
        print(f"    {desc}: {count}")

# Transaction types present in one ledger but missing from another
print("\n  TRANSACTION TYPE COVERAGE:")
all_descs = set()
src_descs = {}
for src in v6["source_id"].unique():
    descs = set(v6[v6["source_id"] == src]["transaction_description"].dropna().unique())
    src_descs[src] = descs
    all_descs |= descs

for desc in sorted(all_descs):
    present_in = [s for s, d in src_descs.items() if desc in d]
    missing_from = [s for s in src_descs.keys() if s not in present_in]
    if missing_from:
        test2_discrepancies.append({
            "type": "MISSING_TRANSACTION_TYPE",
            "description": desc,
            "present_in": ", ".join(present_in),
            "missing_from": ", ".join(missing_from),
        })

print(f"\n  Transaction types unique to specific ledgers: {len(test2_discrepancies)}")
for d in test2_discrepancies[:15]:
    print(f"    '{d['description']}' in [{d['present_in']}] NOT in [{d['missing_from']}]")

# Total amounts per ledger
print("\n  TOTAL AMOUNTS BY LEDGER:")
for src in sorted(v6["source_id"].unique()):
    src_data = v6[v6["source_id"] == src]
    total = src_data["transaction_amount_clean"].sum()
    pos = src_data[src_data["transaction_amount_clean"] > 0]["transaction_amount_clean"].sum()
    neg = src_data[src_data["transaction_amount_clean"] < 0]["transaction_amount_clean"].sum()
    print(f"    {src}: Total={total:>12,.2f}  Positive={pos:>12,.2f}  Negative={neg:>12,.2f}")

# Escrow balance comparison
print("\n  ESCROW BALANCE COMPARISON:")
for src in sorted(v6["source_id"].unique()):
    src_data = v6[v6["source_id"] == src]
    esc = src_data["escrow_balance_clean"].dropna()
    if len(esc) > 0:
        print(f"    {src}: min=${esc.min():>10,.2f}  max=${esc.max():>10,.2f}  "
              f"last=${esc.iloc[-1]:>10,.2f}  count={len(esc)}")

print(f"\n>>> TEST 2 COMPLETE: {len(test2_discrepancies)} discrepancies")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 3: HAF GHOST ALLOCATION
# Did the $35K actually apply to P/I/E or vanish into suspense?
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 3: HAF GHOST ALLOCATION")
print("=" * 72)

test3_findings = []

haf_keywords = ["HAF", "housing assistance", "hardest hit", "homeowner assistance"]
haf_mask = v6["transaction_description"].str.contains(
    "|".join(haf_keywords), case=False, na=False
) | (v6["transaction_amount_clean"] == 35000.0)

haf_rows = v6[haf_mask]
print(f"  HAF-related transactions: {len(haf_rows)}")

for src in haf_rows["source_id"].unique():
    src_haf = haf_rows[haf_rows["source_id"] == src]
    for _, row in src_haf.iterrows():
        principal = float(row.get("principal_paid_clean", 0)) if pd.notna(row.get("principal_paid_clean")) else 0
        interest = float(row.get("interest_paid_clean", 0)) if pd.notna(row.get("interest_paid_clean")) else 0
        escrow = float(row.get("escrow_paid_clean", 0)) if pd.notna(row.get("escrow_paid_clean")) else 0
        suspense = float(row.get("suspense_balance_clean", 0)) if pd.notna(row.get("suspense_balance_clean")) else 0
        total_applied = principal + interest + escrow
        amt = float(row.get("transaction_amount_clean", 0)) if pd.notna(row.get("transaction_amount_clean")) else 0

        if amt > 0 and suspense > 0:
            test3_findings.append({
                "source_id": src,
                "row_id": row["row_id"],
                "amount": amt,
                "principal_applied": principal,
                "interest_applied": interest,
                "escrow_applied": escrow,
                "suspense": suspense,
                "ghost_amount": amt - total_applied,
                "finding": "GHOST" if total_applied == 0 else "PARTIAL",
            })
            print(f"  {src} row {row['row_id']}: ${amt:,.2f} -> "
                  f"P=${principal:,.2f} I=${interest:,.2f} E=${escrow:,.2f} "
                  f"SUSPENSE=${suspense:,.2f}  "
                  f"{'GHOST ALLOCATION' if total_applied == 0 else f'PARTIAL: ${amt - total_applied:,.2f} unaccounted'}")

print(f"\n>>> TEST 3 COMPLETE: {len(test3_findings)} ghost allocations found")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 3b: HAF GHOST ALLOCATION — CROSS-LEDGER BALANCE CHECK
# Compare principal/escrow balances across L1, L2R, L3, L4
# during the HAF period to see if the books match
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 3b: HAF GHOST ALLOCATION — CROSS-LEDGER BALANCE CHECK")
print("During HAF period, do the ledgers agree on where the money went?")
print("=" * 72)

def sf(val):
    """Safe float from any cell value."""
    if pd.isna(val):
        return 0.0
    try:
        return float(str(val).replace(",", "").replace("$", ""))
    except:
        return 0.0

# ── Step 1: Find the HAF deposit date on each ledger ──
print("\n── HAF Deposit: How each ledger recorded $35,000 + $6,464.66 ──\n")

haf_rows = v6[
    (v6["transaction_amount_clean"] == 35000.0) |
    (v6["transaction_amount_clean"] == 6464.66)
].sort_values(["source_id", "row_id"])

for _, row in haf_rows.iterrows():
    p = sf(row.get("principal_paid_clean"))
    i = sf(row.get("interest_paid_clean"))
    e = sf(row.get("escrow_paid_clean"))
    s = sf(row.get("suspense_balance_clean"))
    pb = sf(row.get("principal_balance_clean"))
    eb = sf(row.get("escrow_balance_clean"))
    amt = sf(row.get("transaction_amount_clean"))
    applied = p + i + e

    print(f"  {row['source_id']:>3} row {row['row_id']:>3} | ${amt:>10,.2f} | "
          f"P={p:>8,.2f}  I={i:>8,.2f}  E={e:>8,.2f} | "
          f"Suspense={s:>10,.2f} | Applied={applied:>8,.2f}")
    if applied == 0 and s > 0:
        print(f"       ^^^ GHOST: ${amt:,.2f} went entirely to suspense, "
              f"$0 to principal/interest/escrow")

# ── Step 2: Track where the money went when it LEFT suspense ──
print("\n── Suspense Applications: Where did the money actually land? ──\n")

app_rows = v6[
    (v6["source_id"] == "L2R") &
    (v6["transaction_description"].astype(str).str.contains("Funds Application", case=False)) &
    (v6["principal_paid_clean"].notna()) &
    (v6["principal_paid_clean"].apply(sf) > 0)
].sort_values("row_id", ascending=False)

total_p = 0
total_i = 0
total_e = 0

print(f"  {'Row':>4} {'Due':>12} {'Processed':>12} {'Principal':>10} "
      f"{'Interest':>10} {'Escrow':>10} {'Suspense':>12}")
print("  " + "-" * 85)

for _, row in app_rows.iterrows():
    p = sf(row.get("principal_paid_clean"))
    i = sf(row.get("interest_paid_clean"))
    e = sf(row.get("escrow_paid_clean"))
    s = sf(row.get("suspense_balance_clean"))
    total_p += p
    total_i += i
    total_e += e
    print(f"  {row['row_id']:>4} {str(row['due_date']):>12} {str(row['process_date']):>12} "
          f"${p:>9,.2f} ${i:>9,.2f} ${e:>9,.2f} ${s:>11,.2f}")

total_applied = total_p + total_i + total_e
haf_total = 41464.66 + 3437.96

print(f"\n  {'TOTALS':>4} {'':>12} {'':>12} ${total_p:>9,.2f} ${total_i:>9,.2f} ${total_e:>9,.2f}")
print(f"\n  Total HAF + modified payment into suspense: ${haf_total:>12,.2f}")
print(f"  Total applied out of suspense:              ${total_applied:>12,.2f}")
print(f"  Difference:                                 ${haf_total - total_applied:>12,.2f}")

# ── Step 3: Cross-ledger principal balance comparison ──
print("\n── Principal Balance: What each ledger shows during HAF period ──\n")

for src in ["L1", "L2R", "L3", "L4"]:
    src_data = v6[
        (v6["source_id"] == src) &
        (v6["principal_balance_clean"].notna())
    ].sort_values("row_id")

    if len(src_data) == 0:
        # L1 uses different column — check raw
        src_data2 = v6[
            (v6["source_id"] == src) &
            (v6["principal_balance"].notna())
        ]
        if len(src_data2) > 0:
            print(f"  {src}: {len(src_data2)} rows with principal_balance (raw, not _clean)")
        else:
            print(f"  {src}: NO principal balance data")
        continue

    print(f"  {src}: {len(src_data)} rows with principal balance")
    for _, row in src_data.iterrows():
        pb = sf(row["principal_balance_clean"])
        desc = str(row.get("transaction_description", ""))[:35]
        print(f"    row {row['row_id']:>4} | due {str(row['due_date']):>12} | "
              f"proc {str(row['process_date']):>12} | "
              f"PB = ${pb:>12,.2f} | {desc}")
    print()

# ── Step 4: Escrow balance comparison during HAF period ──
print("── Escrow Balance: What each ledger shows during HAF period ──\n")

for src in ["L1", "L2R", "L3", "L4"]:
    src_data = v6[
        (v6["source_id"] == src) &
        (v6["escrow_balance_clean"].notna())
    ].sort_values("row_id")

    if len(src_data) == 0:
        print(f"  {src}: NO escrow balance data")
        continue

    # Show last 10 rows to cover the HAF period
    tail = src_data.tail(15)
    print(f"  {src}: (last {len(tail)} escrow entries)")
    for _, row in tail.iterrows():
        eb = sf(row["escrow_balance_clean"])
        desc = str(row.get("transaction_description", ""))[:35]
        print(f"    row {row['row_id']:>4} | due {str(row['due_date']):>12} | "
              f"proc {str(row['process_date']):>12} | "
              f"ESC = ${eb:>10,.2f} | {desc}")
    print()

# ── Step 5: The verdict ──
print("=" * 72)
print("GHOST ALLOCATION VERDICT")
print("=" * 72)
print(f"""
  $41,464.66 HAF funds entered suspense on 03/03/2025 and 03/07/2025.
  When applied out on 03/11/2025, the breakdown was:

    To INTEREST:    ${total_i:>10,.2f}  ({total_i/haf_total*100:.1f}% of funds)
    To ESCROW:      ${total_e:>10,.2f}  ({total_e/haf_total*100:.1f}% of funds)
    To PRINCIPAL:   ${total_p:>10,.2f}  ({total_p/haf_total*100:.1f}% of funds)

  The servicer held the borrower delinquent for 11 months (Apr 2024–Feb 2025),
  then when HAF paid the arrearage, captured {total_i/haf_total*100:.1f}% as interest
  on a delinquency the servicer manufactured by routing payments to suspense.

  L3 principal balance: $481,183.37 (02/2024) → $474,073.20 (02/2025)
  Only ${total_p:,.2f} of the ${haf_total:,.2f} reduced the principal.
""")
print(">>> TEST 3b COMPLETE")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 4: DELAYED-POSTING SPREAD
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 4: DELAYED-POSTING SPREAD")
print("=" * 72)

test4_findings = []
v6_with_dates = v6[
    v6["due_date_parsed"].notna() & v6["process_date_parsed"].notna()
].copy()

if len(v6_with_dates) > 0:
    v6_with_dates["posting_delay_days"] = (
        v6_with_dates["process_date_parsed"] - v6_with_dates["due_date_parsed"]
    ).dt.days

    delayed = v6_with_dates[v6_with_dates["posting_delay_days"] > 30]
    print(f"  Transactions with >30 day posting delay: {len(delayed)}")
    print(f"  Total transactions with parseable dates: {len(v6_with_dates)}")

    if len(delayed) > 0:
        print(f"\n  {'row_id':>6} {'src':>4} {'due':>12} {'process':>12} "
              f"{'delay':>6} {'description':>30} {'amount':>12}")
        print("  " + "-" * 96)
        for _, row in delayed.sort_values("posting_delay_days", ascending=False).head(20).iterrows():
            desc = str(row["transaction_description"])[:30]
            amt = float(row.get("transaction_amount_clean", 0)) if pd.notna(row.get("transaction_amount_clean")) else 0
            test4_findings.append({
                "row_id": row["row_id"],
                "source_id": row["source_id"],
                "due_date": str(row["due_date"]),
                "process_date": str(row["process_date"]),
                "delay_days": row["posting_delay_days"],
                "description": row["transaction_description"],
                "amount": amt,
            })
            print(f"  {row['row_id']:>6} {row['source_id']:>4} "
                  f"{str(row['due_date']):>12} {str(row['process_date']):>12} "
                  f"{row['posting_delay_days']:>6} {desc:>30} {amt:>12,.2f}")

    # Visualization
    plot_data = v6_with_dates[
        (v6_with_dates["process_date_parsed"] >= pd.Timestamp("2020-01-01")) &
        (v6_with_dates["process_date_parsed"] <= pd.Timestamp("2030-01-01")) &
        v6_with_dates["posting_delay_days"].notna()
    ].copy()
    fig, ax = plt.subplots(figsize=(12, 6))
    for src in plot_data["source_id"].unique():
        src_data = plot_data[plot_data["source_id"] == src]
        if len(src_data) > 0:
            ax.scatter(
                src_data["process_date_parsed"],
                src_data["posting_delay_days"],
                label=src, alpha=0.6, s=40,
            )
    ax.axhline(y=0, color="green", linewidth=1, linestyle="--", label="On-time")
    ax.axhline(y=30, color="orange", linewidth=1, linestyle="--", label="30-day threshold")
    ax.set_title("Posting Delay Analysis (Process Date - Due Date)", fontweight="bold")
    ax.set_ylabel("Delay (days)")
    ax.set_xlabel("Process Date")
    ax.legend()
    plt.tight_layout()
    plt.show()

print(f"\n>>> TEST 4 COMPLETE: {len(test4_findings)} delayed transactions")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 5: CONTRADICTORY SUSPENSE ALLOCATIONS
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 5: CONTRADICTORY SUSPENSE ALLOCATIONS")
print("=" * 72)

test5_findings = []

susp_rows = v6[v6["suspense_balance_clean"].notna()].copy()
susp_by_src = {}
for src in susp_rows["source_id"].unique():
    susp_by_src[src] = susp_rows[susp_rows["source_id"] == src]

# Compare suspense entries across ledgers for same due_date
for src1 in susp_by_src:
    for src2 in susp_by_src:
        if src1 >= src2:
            continue
        df1 = susp_by_src[src1]
        df2 = susp_by_src[src2]

        for _, r1 in df1.iterrows():
            matches = df2[
                (df2["transaction_amount_clean"] == r1["transaction_amount_clean"]) &
                (df2["due_date"] == r1["due_date"])
            ]
            for _, r2 in matches.iterrows():
                s1 = r1["suspense_balance_clean"]
                s2 = r2["suspense_balance_clean"]
                if abs(s1 - s2) > 0.01:
                    test5_findings.append({
                        "due_date": r1["due_date"],
                        "amount": r1["transaction_amount_clean"],
                        f"suspense_{src1}": s1,
                        f"suspense_{src2}": s2,
                        "difference": abs(s1 - s2),
                        "src1_row": r1["row_id"],
                        "src2_row": r2["row_id"],
                    })
                    print(f"  CONTRADICTION: due={r1['due_date']} amt=${r1['transaction_amount_clean']:,.2f}  "
                          f"{src1}=${s1:,.2f} vs {src2}=${s2:,.2f}  "
                          f"diff=${abs(s1 - s2):,.2f}")

if not test5_findings:
    print("  Suspense values consistent across ledgers (amounts match where comparable)")

# Net suspense per source
for src in susp_by_src:
    net = susp_by_src[src]["suspense_balance_clean"].sum()
    print(f"  {src} net suspense: ${net:,.2f}")
    if abs(net) > 0.01:
        test5_findings.append({
            "type": "NET_IMBALANCE",
            "source_id": src,
            "net_suspense": net,
        })

print(f"\n>>> TEST 5 COMPLETE: {len(test5_findings)} findings")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 6: SUSPENSE ACCOUNT KITING
# Payments routed to suspense instead of being applied to loan
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 6: SUSPENSE ACCOUNT KITING")
print("Payments routed to suspense instead of being applied to loan")
print("=" * 72)

test6_findings = []

payment_rows = v6[
    v6["transaction_description"].str.contains("PAYMENT|Payment|Funds Application", case=False, na=False)
].copy()

kited = payment_rows[
    (payment_rows["suspense_balance_clean"].notna()) &
    (payment_rows["suspense_balance_clean"] > 0) &
    (payment_rows["transaction_amount_clean"] > 0)
]

print(f"  Payments routed to suspense: {len(kited)}")
total_kited = 0
for _, row in kited.iterrows():
    amt = float(row.get("transaction_amount_clean", 0)) if pd.notna(row.get("transaction_amount_clean")) else 0
    susp = float(row.get("suspense_balance_clean", 0)) if pd.notna(row.get("suspense_balance_clean")) else 0
    total_kited += susp
    test6_findings.append({
        "row_id": row["row_id"],
        "source_id": row["source_id"],
        "due_date": row["due_date"],
        "process_date": row["process_date"],
        "payment_amount": amt,
        "to_suspense": susp,
        "description": row["transaction_description"],
    })
    print(f"    {row['source_id']} row {row['row_id']}: "
          f"${amt:>10,.2f} -> suspense ${susp:>10,.2f}  "
          f"due={row['due_date']}  process={row['process_date']}")

print(f"\n  TOTAL KITED TO SUSPENSE: ${total_kited:,.2f}")
print(f"\n>>> TEST 6 COMPLETE: {len(test6_findings)} kited payments")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 7: ESCROW ADVANCE VELOCITY
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 7: ESCROW ADVANCE VELOCITY")
print("=" * 72)

test7_findings = []

esc_adv = v6[
    v6["transaction_description"].str.contains(
        "Escrow Advance|ESCROW ADVANCE|escrow advance", case=False, na=False
    ) & ~v6["transaction_description"].str.contains("Repay|REPAY", case=False, na=False)
]

print(f"  Escrow advances found: {len(esc_adv)}")
total_advances = 0
for _, row in esc_adv.iterrows():
    amt = abs(float(row.get("transaction_amount_clean", 0))) if pd.notna(row.get("transaction_amount_clean")) else 0
    esc = float(row.get("escrow_paid_clean", 0)) if pd.notna(row.get("escrow_paid_clean")) else 0
    total_advances += max(amt, abs(esc))
    desc = str(row["transaction_description"])[:40]
    test7_findings.append({
        "row_id": row["row_id"],
        "source_id": row["source_id"],
        "due_date": row["due_date"],
        "process_date": row["process_date"],
        "amount": amt,
        "escrow_applied": esc,
        "description": row["transaction_description"],
    })
    print(f"    {row['source_id']} row {row['row_id']}: "
          f"${amt:>10,.2f}  escrow=${esc:>10,.2f}  {desc}")

print(f"\n  TOTAL ESCROW ADVANCES: ${total_advances:,.2f}")

# Repayments
esc_repay = v6[
    v6["transaction_description"].str.contains(
        "Escrow Advance Rep|REPAY OF ESCROW|escrow advance rep", case=False, na=False
    )
]
total_repaid = 0
for _, row in esc_repay.iterrows():
    esc = abs(float(row.get("escrow_paid_clean", 0))) if pd.notna(row.get("escrow_paid_clean")) else 0
    total_repaid += esc

print(f"  Total escrow advance repayments: ${total_repaid:,.2f}")
print(f"  Net escrow advance exposure: ${total_advances - total_repaid:,.2f}")
print(f"\n>>> TEST 7 COMPLETE: {len(test7_findings)} advances")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 8: ROUND-NUMBER FEE DETECTION
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 8: ROUND-NUMBER FEE DETECTION")
print("=" * 72)

test8_findings = []

fee_rows = v6[
    v6["transaction_description"].str.contains(
        "Fee|FEE|Charge|CHARGE|Advance|ADVANCE|Corp Adv|ATTORNEY",
        case=False, na=False,
    )
].copy()

print(f"  Fee-related transactions: {len(fee_rows)}")

for _, row in fee_rows.iterrows():
    amt = abs(row.get("transaction_amount_clean", 0) or 0)
    other = abs(row.get("other_amount_clean", 0) or 0)
    check_amt = amt if amt > 0 else other

    if check_amt > 0:
        is_round = (check_amt % 25 == 0) or (check_amt % 50 == 0) or (check_amt % 100 == 0)
        test8_findings.append({
            "row_id": row["row_id"],
            "source_id": row["source_id"],
            "description": row["transaction_description"],
            "amount": check_amt,
            "is_round_number": is_round,
            "due_date": row["due_date"],
        })
        if is_round and check_amt > 0:
            print(f"    ROUND: {row['source_id']} row {row['row_id']}: "
                  f"${check_amt:>10,.2f}  {row['transaction_description']}")

if test8_findings:
    df8 = pd.DataFrame(test8_findings)
    round_count = df8[df8["is_round_number"]].shape[0]
    total_count = len(df8[df8["amount"] > 0])
    print(f"\n  Round-number fees: {round_count}/{total_count} "
          f"({round_count / total_count * 100:.1f}% if fees are round)")

print(f"\n>>> TEST 8 COMPLETE: {len(test8_findings)} fee entries analyzed")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 9: BENFORD'S LAW — LEADING DIGIT ANALYSIS
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 9: BENFORD'S LAW — LEADING DIGIT ANALYSIS")
print("=" * 72)

amounts = v6["transaction_amount_clean"].dropna()
amounts = amounts[amounts.abs() > 0]

leading_digits = amounts.abs().apply(lambda x: int(str(f"{x:.10f}").lstrip("0").lstrip(".")[0]))
observed = leading_digits.value_counts().sort_index()

benford_expected = {d: np.log10(1 + 1 / d) for d in range(1, 10)}
total = len(leading_digits)

print(f"  Total non-zero transactions: {total}")
print(f"\n  {'Digit':>5} {'Observed':>10} {'Expected':>10} {'Obs%':>8} {'Exp%':>8} {'Deviation':>10}")
print("  " + "-" * 55)

test9_deviations = []
for d in range(1, 10):
    obs_count = observed.get(d, 0)
    obs_pct = obs_count / total * 100
    exp_pct = benford_expected[d] * 100
    dev = obs_pct - exp_pct
    test9_deviations.append({
        "digit": d,
        "observed_count": obs_count,
        "expected_pct": exp_pct,
        "observed_pct": obs_pct,
        "deviation_pct": dev,
    })
    flag = " ***" if abs(dev) > 5 else ""
    print(f"  {d:>5} {obs_count:>10} {total * benford_expected[d]:>10.1f} "
          f"{obs_pct:>7.1f}% {exp_pct:>7.1f}% {dev:>+9.1f}%{flag}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
digits = range(1, 10)
obs_pcts = [observed.get(d, 0) / total * 100 for d in digits]
exp_pcts = [benford_expected[d] * 100 for d in digits]

x = np.arange(len(digits))
width = 0.35
ax.bar(x - width / 2, obs_pcts, width, label="Observed", color="steelblue")
ax.bar(x + width / 2, exp_pcts, width, label="Benford Expected", color="coral")
ax.set_xlabel("Leading Digit")
ax.set_ylabel("Frequency (%)")
ax.set_title("Benford's Law Analysis — Transaction Amounts", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(digits)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n>>> TEST 9 COMPLETE: {len(test9_deviations)} digits analyzed")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 10: PRINCIPAL BALANCE CONTINUITY CHECK
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 10: PRINCIPAL BALANCE CONTINUITY CHECK")
print("=" * 72)

test10_findings = []

for src in v6["source_id"].unique():
    src_data = v6[v6["source_id"] == src].sort_values("row_id")
    balances = src_data[src_data["principal_balance_clean"].notna()]

    if len(balances) < 2:
        continue

    print(f"\n  {src}: {len(balances)} rows with principal balance")
    prev_bal = None
    for _, row in balances.iterrows():
        bal = row["principal_balance_clean"]
        paid = float(row.get("principal_paid_clean", 0)) if pd.notna(row.get("principal_paid_clean")) else 0

        if prev_bal is not None:
            expected = prev_bal - paid
            gap = bal - expected
            if abs(gap) > 0.01 and paid > 0:
                test10_findings.append({
                    "source_id": src,
                    "row_id": row["row_id"],
                    "prev_balance": prev_bal,
                    "principal_paid": paid,
                    "expected_balance": expected,
                    "actual_balance": bal,
                    "gap": gap,
                })
                if abs(gap) > 100:
                    print(f"    row {row['row_id']}: prev=${prev_bal:,.2f} "
                          f"paid=${paid:,.2f} expected=${expected:,.2f} "
                          f"actual=${bal:,.2f} GAP=${gap:,.2f}")
        prev_bal = bal

print(f"\n  Principal continuity gaps found: {len(test10_findings)}")
print(f"\n>>> TEST 10 COMPLETE")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEST 11: LATE FEE COMPUTATION CHECK
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("TEST 11: LATE FEE COMPUTATION CHECK")
print("=" * 72)

test11_findings = []

late_fees = v6[
    v6["transaction_description"].str.contains(
        "Late Charge|LATE CHARGE", case=False, na=False
    )
].copy()

print(f"  Late charge entries: {len(late_fees)}")

expected_late_fee = 133.27  # Per MASTER 07_TEST_Fees
print(f"  Expected late fee per note terms: ${expected_late_fee:,.2f}")

for _, row in late_fees.iterrows():
    other = abs(float(row.get("other_amount_clean", 0))) if pd.notna(row.get("other_amount_clean")) else 0
    amt = abs(float(row.get("transaction_amount_clean", 0))) if pd.notna(row.get("transaction_amount_clean")) else 0
    fee = other if other > 0 else amt

    if fee > 0:
        variance = fee - expected_late_fee
        test11_findings.append({
            "row_id": row["row_id"],
            "source_id": row["source_id"],
            "due_date": row["due_date"],
            "late_fee_charged": fee,
            "expected_fee": expected_late_fee,
            "variance": variance,
        })
        if abs(variance) > 0.01:
            print(f"    {row['source_id']} row {row['row_id']}: "
                  f"charged=${fee:,.2f} expected=${expected_late_fee:,.2f} "
                  f"variance=${variance:+,.2f}")

# Count matching fees
fee_133 = late_fees[late_fees["other_amount_clean"].abs().between(133.26, 133.28) |
                    late_fees["transaction_amount_clean"].abs().between(133.26, 133.28)]
print(f"\n  $133.27 late fees (matching expected): {len(fee_133)}")

print(f"\n>>> TEST 11 COMPLETE: {len(test11_findings)} late fee entries checked")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DAMAGES SUMMARY — FULL FRAMEWORK
# Run AFTER all 11 tests above
# ══════════════════════════════════════════════════════════════

print("=" * 72)
print("DAMAGES SUMMARY — FULL FRAMEWORK")
print("=" * 72)

damages = []
daily_rate = NOTE_RATE / 365

# ── A. ACTUAL DAMAGES ─────────────────────────────────────────

# A1. Wrongful interest accrual from principal balance inflation
haf_total = 41464.66  # $35,000 + $6,464.66
months_delinquent = 11  # April 2024 through February 2025
monthly_interest_overcharge = haf_total * (NOTE_RATE / 12)
wrongful_interest = monthly_interest_overcharge * months_delinquent
damages.append({
    "category": "A1. Wrongful Interest Accrual",
    "violation": "\u00a71026.36(c)(1)(ii)(C)",
    "status": "ADMITTED",
    "principal_amount": haf_total,
    "estimated_damage": round(wrongful_interest, 2),
    "basis": f"${haf_total:,.2f} x {NOTE_RATE*100:.1f}% / 12 x {months_delinquent} months unapplied",
})

# A2. Principal balance continuity gaps
l3_gaps = [f for f in test10_findings if f.get("source_id") == "L3"]
total_gap = sum(abs(f["gap"]) for f in l3_gaps)
damages.append({
    "category": "A2. Principal Overstatement",
    "violation": "\u00a71026.36(c)(1)(ii)(C)",
    "status": "SUPPORTED",
    "principal_amount": total_gap,
    "estimated_damage": round(total_gap, 2),
    "basis": f"{len(l3_gaps)} months of balance inflation totaling ${total_gap:,.2f} (L3)",
})

# A3. Late fees wrongfully assessed
late_fees_v6 = v6[v6["transaction_description"].str.contains(
    "Late Charge Assess|LATE CHARGE ASSESS", case=False, na=False
)]
l3_late = late_fees_v6[late_fees_v6["source_id"] == "L3"]
l2r_late = late_fees_v6[late_fees_v6["source_id"] == "L2R"]
unique_late_count = max(len(l3_late), len(l2r_late))
total_late_fees = unique_late_count * 133.27
damages.append({
    "category": "A3. Wrongful Late Fees",
    "violation": "\u00a71026.36(c)(1)(ii)(C)",
    "status": "SUPPORTED",
    "principal_amount": total_late_fees,
    "estimated_damage": round(total_late_fees, 2),
    "basis": f"{unique_late_count} late charges x $133.27 assessed during manufactured delinquency",
})

# A4. Foreclosure/attorney advances
fc_adv = v6[v6["transaction_description"].str.contains(
    "Attorney|ATTORNEY|Statutory|STATUTORY", case=False, na=False
)]
l3_fc = fc_adv[fc_adv["source_id"] == "L3"]
total_fc = 0
for _, row in l3_fc.iterrows():
    amt = abs(float(row.get("transaction_amount_clean", 0))) if pd.notna(row.get("transaction_amount_clean")) else 0
    total_fc += amt
damages.append({
    "category": "A4. Foreclosure/Attorney Costs",
    "violation": "\u00a72605 (RESPA)",
    "status": "SUPPORTED",
    "principal_amount": total_fc,
    "estimated_damage": round(total_fc, 2),
    "basis": "Attorney advances + statutory expenses charged during suspense hold",
})

# A5. Escrow advance exposure
esc_advances = 0
esc_repaid = 0
for _, row in v6[v6["source_id"] == "L3"].iterrows():
    desc = str(row.get("transaction_description", ""))
    esc_val = float(row.get("escrow_paid_clean", 0)) if pd.notna(row.get("escrow_paid_clean")) else 0
    if "ESCROW ADVANCE" in desc and "REPAY" not in desc and esc_val > 0:
        esc_advances += esc_val
    elif "REPAY OF ESCROW" in desc and esc_val < 0:
        esc_repaid += abs(esc_val)
net_escrow = esc_advances - esc_repaid
if net_escrow > 0:
    damages.append({
        "category": "A5. Net Escrow Advance Exposure",
        "violation": "\u00a72605 (RESPA)",
        "status": "SUPPORTED",
        "principal_amount": net_escrow,
        "estimated_damage": round(net_escrow, 2),
        "basis": f"${esc_advances:,.2f} advanced - ${esc_repaid:,.2f} repaid (L3)",
    })

# A6. Admitted misapplication
damages.append({
    "category": "A6. Admitted Misapplication",
    "violation": "\u00a71026.36(c)(1)(ii)(C)",
    "status": "ADMITTED",
    "principal_amount": 1350.69,
    "estimated_damage": 1350.69,
    "basis": "Misapplication reversal of $1,350.69 confirmed in L2R, L3, L4",
})

# ── B. STATUTORY DAMAGES ──────────────────────────────────────

damages.append({
    "category": "B1. RESPA \u00a72605(f) Statutory",
    "violation": "12 USC \u00a72605(f)(1)(A)",
    "status": "ADMITTED",
    "principal_amount": 0,
    "estimated_damage": 2000.00,
    "basis": "Pattern/practice: admitted suspense kiting + 22-row treadmill + 3 misapplication reversals",
})
damages.append({
    "category": "B2. TILA \u00a71640(a) Statutory",
    "violation": "15 USC \u00a71640(a)(2)(A)",
    "status": "SUPPORTED",
    "principal_amount": 0,
    "estimated_damage": 4000.00,
    "basis": "Statutory damages for \u00a71026.36(c)(1)(ii)(C) violation (2x finance charge, min $400, max $4,000)",
})
damages.append({
    "category": "B3. FDCPA \u00a71692k Statutory",
    "violation": "15 USC \u00a71692k(a)(2)(A)",
    "status": "CONDITIONAL",
    "principal_amount": 0,
    "estimated_damage": 1000.00,
    "basis": "Statutory damages if foreclosure notices constitute debt collection",
})
damages.append({
    "category": "B4. FCRA Credit Reporting",
    "violation": "15 USC \u00a71681s-2",
    "status": "CONDITIONAL",
    "principal_amount": 0,
    "estimated_damage": 1000.00,
    "basis": "Statutory damages for reporting delinquency created by servicer's own suspense routing",
})

# ── C. CONSEQUENTIAL ──────────────────────────────────────────

damages.append({
    "category": "C1. Emotional Distress",
    "violation": "State tort / \u00a72605",
    "status": "CONDITIONAL",
    "principal_amount": 0,
    "estimated_damage": 25000.00,
    "basis": "Conservative estimate: foreclosure threat while $41,464.66 held in suspense (discovery-dependent)",
})
damages.append({
    "category": "C2. Credit Score Actual Damages",
    "violation": "FCRA \u00a71681s-2",
    "status": "CONDITIONAL",
    "principal_amount": 0,
    "estimated_damage": 10000.00,
    "basis": "Estimated credit damage from 11-month manufactured delinquency (discovery-dependent)",
})

# ── D. FEE-SHIFTING ───────────────────────────────────────────

damages.append({
    "category": "D1. Attorney's Fees",
    "violation": "\u00a72605(f)(3) / \u00a71640(a)(3)",
    "status": "MANDATORY",
    "principal_amount": 0,
    "estimated_damage": 0,
    "basis": "Mandatory fee-shifting under RESPA \u00a72605(f)(3), TILA \u00a71640(a)(3), FDCPA \u00a71692k(a)(3)",
})

# ── PRINT SUMMARY ─────────────────────────────────────────────

damages_df = pd.DataFrame(damages)

actual = damages_df[damages_df["category"].str.startswith("A")]
statutory = damages_df[damages_df["category"].str.startswith("B")]
consequential = damages_df[damages_df["category"].str.startswith("C")]

actual_total = actual["estimated_damage"].sum()
statutory_total = statutory["estimated_damage"].sum()
consequential_total = consequential["estimated_damage"].sum()

print("\n  === A. ACTUAL DAMAGES (From Ledger Data) ===")
print(f"  {'Category':<40} {'Status':<12} {'Amount':>12} {'Violation'}")
print("  " + "-" * 90)
for _, d in actual.iterrows():
    print(f"  {d['category']:<40} {d['status']:<12} "
          f"${d['estimated_damage']:>10,.2f} {d['violation']}")
print(f"  {'SUBTOTAL ACTUAL':<40} {'':12} ${actual_total:>10,.2f}")

print("\n  === B. STATUTORY DAMAGES ===")
print(f"  {'Category':<40} {'Status':<12} {'Amount':>12} {'Violation'}")
print("  " + "-" * 90)
for _, d in statutory.iterrows():
    print(f"  {d['category']:<40} {d['status']:<12} "
          f"${d['estimated_damage']:>10,.2f} {d['violation']}")
print(f"  {'SUBTOTAL STATUTORY':<40} {'':12} ${statutory_total:>10,.2f}")

print("\n  === C. CONSEQUENTIAL (Discovery-Dependent) ===")
print(f"  {'Category':<40} {'Status':<12} {'Amount':>12} {'Violation'}")
print("  " + "-" * 90)
for _, d in consequential.iterrows():
    print(f"  {d['category']:<40} {d['status']:<12} "
          f"${d['estimated_damage']:>10,.2f} {d['violation']}")
print(f"  {'SUBTOTAL CONSEQUENTIAL':<40} {'':12} ${consequential_total:>10,.2f}")

print("\n  === D. FEE-SHIFTING ===")
print("  D1. Attorney's Fees                     MANDATORY     per statute  RESPA/TILA/FDCPA")

grand_total = actual_total + statutory_total + consequential_total
print("\n  " + "=" * 90)
print(f"  {'TOTAL (A+B+C, excl. atty fees)':<40} {'':12} ${grand_total:>10,.2f}")
print(f"  {'  Admitted/Supported floor (A+B)':<40} {'':12} ${actual_total + statutory_total:>10,.2f}")
print(f"  {'  Full claim incl. consequential':<40} {'':12} ${grand_total:>10,.2f}")
print(f"\n  Note: Treble damages under Wyoming consumer protection (Wyo. Stat. \u00a740-12-108)")
print(f"  could multiply actual damages to ${actual_total * 3:>10,.2f}")
print(f"  Attorney's fees are mandatory fee-shifting, not included in totals above.")

print(f"\n>>> DAMAGES FRAMEWORK COMPLETE")
display(damages_df)